# Marketplace Safety - Prioritization of Suspicious Listings and Messages

**Background**
This project was carried out on behalf of the analytics team at a company that operates a 
marketplace app, similar to Blocket. The platform is used daily by a large number of users, the vast majority of whom are genuine. However, every week a small proportion of problematic activity occurs: scam listings, spam, suspicious accounts acting quickly, and attempts to move conversations off the platform.

**Problem**
The company's Trust & Safety team manually reviews and handles suspicious content. The problem is that the volume is too large for the team to review everything in time. Management is therefore requesting a decision-support tool that helps the team prioritize what to review first.

**Goal**
The goal is to deliver a solution that:
- performs reasonably well on new, unseen data
- can be run regularly in production
- can be explained to non-technical stakeholders

The focus is on prioritization and decision support.

**Stakeholder**


Kolumnnamn | Förklaring |
|---|---|
| `id` | Unikt identifieringsnummer för varje post |
| `day` | Dagen då händelsen inträffade |
| `event_type` | Typ av händelse (t.ex. visning, meddelande, anmälan) |
| `category` | Produktkategori för annonsen |
| `region` | Geografiskt område där annonsen publicerades |
| `device` | Enhetstyp som användes (t.ex. mobil, dator) |
| `account_age_days` | Antal dagar sedan kontot skapades |
| `num_prev_listings` | Antal tidigare annonser från samma användare |
| `prev_reports_30d` | Antal gånger användaren anmälts de senaste 30 dagarna |
| `verification_level` | Verifieringsnivå för kontot |
| `price` | Annonsens pris |
| `num_images` | Antal bilder i annonsen |
| `message_length` | Längden på meddelandet i annonsen |
| `contains_off_platform` | Om användaren försökt flytta konversationen utanför plattformen |
| `urgency_words` | Om annonsen innehåller ord som skapar artificiell brådska |
| `payment_attempt` | Om ett betalningsförsök har skett |
| `time_to_first_response_min` | Tid i minuter till första svar |
| `is_suspicious` | Målvariabel — om annonsen är misstänkt (1) eller inte (0) |

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from joblib import dump, load

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import (
    accuracy_score, 
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(palette="rocket", font="Helvetica")

1) Data understanding & EDA (brief but relevant)

- Show dataset size, data types and target distribution.
- Check for missing values and describe how you handle them.
- At least 2 figures/tables + brief interpretation.

2) Train/test + preprocessing (correct workflow)

- Create a train/test split from historical_data.csv.
- Build a pipeline where preprocessing is done in a way that prevents test data from influencing training (avoid leakage).
- For classification: use stratified split so classes are distributed reasonably.

3) Modeling and comparison

- Create a baseline.
- Train at least two additional models (at least 3 total including baseline).
- Evaluate with a reasonable method (e.g. cross-validation on train or a clear validation setup).
- Choose a metric and justify the choice based on your requirements card.
(Example models: LogisticRegression, DecisionTree, RandomForest, GradientBoosting…)

4) Select and optimize ONE model (hyperparameter tuning)

- Select a "final" model based on the comparison.
- Tune the selected model (small grid, at least 1–2 parameters).
- Briefly explain what you optimized and why (connect to the requirements card).

5) Threshold / prioritization (linked to the requirements card)
- You must decide how the model will be used in practice. Choose a strategy:

    A) Threshold decision — flag as suspicious if probability ≥ t. Justify t based on the requirements card and show consequences (FP/FN or precision/recall) 
    B) Top-X prioritization — flag the X% highest risk (e.g. top 5% or top 50 per day). Justify X based on the requirements card and show consequences.

You must clearly show how your choice affects FP/FN and why it suits your stakeholder.

6) Deploy test: new data (Tuesday course week 6)
When you receive new_data.csv you should:

- use your locked pipeline
- create predictions and a priority list